# Week 4, Day 4: Linear combinations, span and basis: the population vector

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w004/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
v1 = np.array([2.0, 0.5]); v2 = np.array([0.5, 1.5])
c = np.array([1.5, 1.0])
by_hand = c[0] * v1 + c[1] * v2
V = np.column_stack([v1, v2])         # vectors as columns, shape (2, 2)
print("by hand:", by_hand, "  as V @ c:", V @ c)

### Exercise: Independent or not?

Write `independent(vectors)` that returns True if the list of vectors is linearly independent. Stack them as columns and compare the rank with the number of vectors.

In [ ]:
def independent(vectors):
    ...

print(independent([np.array([1.0, 2.0]), np.array([2.0, 4.0])]))

In [ ]:
import numpy as np
def test_cases():
    "independent gives the right answers"
    assert independent([np.array([1.0, 0.0]), np.array([0.0, 1.0])])
    assert not independent([np.array([1.0, 2.0]), np.array([2.0, 4.0])])
    assert independent([np.array([1.0, 0, 0]), np.array([1.0, 1, 0]), np.array([1.0, 1, 1])])
    assert not independent([np.array([1.0, 0, 0]), np.array([0, 1.0, 0]), np.array([1.0, 1.0, 0])])
    assert not independent([np.array([1.0, 0]), np.array([0, 1.0]), np.array([1.0, 1.0])])

test_cases()
print("All checks passed")

### Exercise: Coordinates in a new basis

Write `coords(w, u, v)` that returns the coefficients $(c_1, c_2)$ of `w` in the basis `u`, `v`, using `np.linalg.solve`.

In [ ]:
def coords(w, u, v):
    ...

print(coords(np.array([3.5, 2.25]), np.array([2.0, 0.5]), np.array([0.5, 1.5])))

In [ ]:
import numpy as np
def test_coords():
    "coords solves V c = w"
    u, v, w = np.array([2.0, 0.5]), np.array([0.5, 1.5]), np.array([3.5, 2.25])
    c = coords(w, u, v)
    assert np.allclose(c[0] * u + c[1] * v, w)
    assert np.allclose(c, [1.5, 1.0])

test_coords()
print("All checks passed")

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def simulate_rates(direction_deg, prefs_deg, b0=20.0, k=15.0, T=0.5, rng=rng):
    """Poisson spike counts in a window of T seconds, converted to rates (Hz)."""
    d = np.radians(direction_deg); p = np.radians(prefs_deg)
    mean = (b0 + k * np.cos(d - p)) * T
    return rng.poisson(mean) / T

prefs = rng.uniform(0, 360, 100)
r = simulate_rates(70, prefs)
print("first ten rates:", np.round(r[:10], 1))

### Exercise: A population-vector decoder

Write `pop_vector(rates, prefs_deg, b0=20, k=15)` that returns the decoded direction in degrees (0 to 360) using the formula above. Then, for population sizes `sizes`, decode 200 random movement directions each and store the mean absolute angular error (degrees) in `errors`. Plot error against population size.

In [ ]:
def pop_vector(rates, prefs_deg, b0=20.0, k=15.0):
    w = ...
    p = np.radians(prefs_deg)
    x, y = ..., ...
    return np.degrees(np.arctan2(y, x)) % 360

def ang_err(a, b):
    return np.abs((a - b + 180) % 360 - 180)

sizes = [5, 10, 20, 50, 100, 200, 500]
errors = []
for n in sizes:
    errs = []
    for _ in range(200):
        prefs = rng.uniform(0, 360, n)
        true = rng.uniform(0, 360)
        errs.append(ang_err(pop_vector(simulate_rates(true, prefs), prefs), true))
    errors.append(np.mean(errs))

plt.figure(figsize=(5, 3)); plt.loglog(sizes, errors, "o-")
plt.xlabel("number of neurons"); plt.ylabel("mean error (deg)"); plt.show()
print(np.round(errors, 1))

In [ ]:
import numpy as np
def test_exact():
    "noise-free rates are decoded exactly"
    prefs = np.arange(0, 360, 10.0)
    r = 20 + 15 * np.cos(np.radians(123 - prefs))
    assert abs(((pop_vector(r, prefs) - 123) + 180) % 360 - 180) < 1e-6

def test_errors():
    "errors shrink as the population grows"
    assert len(errors) == len(sizes)
    assert errors[-1] < errors[0] / 2, f"errors: {np.round(errors, 1)}"

test_exact()
test_errors()
print("All checks passed")

### Exercise: When the population vector is biased

Now give 80 per cent of 200 neurons preferred directions between 0° and 90°, and the rest uniform. Decode a movement at 100°, 300 times, and store the mean signed error (decoded minus true, wrapped to between -180 and 180) in `mean_signed_error`. Which way is the estimate pulled, and why?

In [ ]:
true = 100.0
signed = []
for _ in range(300):
    prefs = ...
    est = pop_vector(simulate_rates(true, prefs), prefs)
    signed.append((est - true + 180) % 360 - 180)
mean_signed_error = np.mean(signed)
print(f"mean signed error: {mean_signed_error:.1f} degrees")

In [ ]:
import numpy as np
def test_bias():
    "mean_signed_error shows a clear bias for clustered preferred directions"
    assert abs(mean_signed_error) > 5, f"mean signed error {mean_signed_error:.1f} degrees"

test_bias()
print("All checks passed")